In [1]:
# =========================================================
# CLIENT LEVEL FEATURE STORE
# =========================================================

import pandas as pd
import numpy as np
import json

In [25]:
# =========================================================
# LOAD DATA
# =========================================================

# Transactions
df_trx = pd.read_csv(r"data\transactions_data.csv")

# Cards
df_card = pd.read_csv(r"data\cards_data.csv")

# Users
df_user = pd.read_csv(r"data\users_data.csv")

# MCC
with open(r"data\mcc_codes.json", "r") as f:
    mcc = json.load(f)

df_mcc = pd.DataFrame(
    list(mcc.items()),
    columns=["mcc_code", "mcc_description"]
)

# Fraud labels
with open(r"data\train_fraud_labels.json", "r") as f:
    fraud = json.load(f)

df_fraud = pd.DataFrame(
    fraud["target"].items(),
    columns=["transaction_id", "fraud_label"]
)


In [26]:

# =========================================================
# COPY DATA
# =========================================================

trx = df_trx.copy()
card = df_card.copy()
mcc = df_mcc.copy()
fraud = df_fraud.copy()

In [27]:
# =========================================================
# DATA CLEANING
# =========================================================

# ---------- ID TYPES ----------
trx["id"] = trx["id"].astype(int)
fraud["transaction_id"] = fraud["transaction_id"].astype(int)

trx["mcc"] = trx["mcc"].astype(str)
mcc["mcc_code"] = mcc["mcc_code"].astype(str)

# ---------- DATE ----------
trx["date"] = pd.to_datetime(trx["date"])

card["acct_open_date"] = pd.to_datetime(
    card["acct_open_date"],
    errors="coerce"
)

# ---------- AMOUNT ----------
trx["amount"] = (
    trx["amount"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

trx["abs_amount"] = trx["amount"].abs()

# ---------- CREDIT LIMIT ----------
card["credit_limit"] = (
    card["credit_limit"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# =========================================
# ERROR DUMMIES
# =========================================

error_dummies = trx["errors"].fillna("").str.get_dummies(sep=",")

# Clean column names
error_dummies.columns = (
    error_dummies.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Merge back
trx = pd.concat(
    [trx, error_dummies],
    axis=1
)

C:\Users\berna\AppData\Local\Temp\ipykernel_7752\3890780132.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  card["acct_open_date"] = pd.to_datetime(


In [28]:

# =========================================================
# JOIN TABLES
# =========================================================

# ---------- FRAUD ----------
trx = trx.merge(
    fraud,
    left_on="id",
    right_on="transaction_id",
    how="left"
)

trx["fraud_label"] = (
    trx["fraud_label"]
    .map({"Yes": 1, "No": 0})
    .fillna(0)
    .astype(int)
)

# ---------- MCC ----------
trx = trx.merge(
    mcc,
    left_on="mcc",
    right_on="mcc_code",
    how="left"
)


In [29]:

# =========================================================
# TIME FEATURES
# =========================================================

trx["year"] = trx["date"].dt.year
trx["month"] = trx["date"].dt.month
trx["year_month"] = trx["date"].dt.to_period("M")

trx["weekday"] = trx["date"].dt.weekday
trx["hour"] = trx["date"].dt.hour

trx["is_weekend"] = (
    trx["weekday"].isin([5, 6])
).astype(int)

trx["is_night"] = (
    (trx["hour"] >= 22) |
    (trx["hour"] <= 5)
).astype(int)

# =========================================================
# CASHFLOW FEATURES
# =========================================================

trx["cashflow_type"] = np.where(
    trx["amount"] < 0,
    "outflow",
    "inflow"
)

# =========================================================
# MONTHLY CLIENT AGGREGATION
# =========================================================

monthly_client = (
    trx.groupby(["client_id", "year_month"])
    .agg(
        monthly_inflow=(
            "amount",
            lambda x: x[x > 0].sum()
        ),

        monthly_outflow=(
            "abs_amount",
            lambda x: x[
                trx.loc[x.index, "cashflow_type"] == "outflow"
            ].sum()
        ),

        monthly_transaction_count=("id", "count"),

        active_days=(
            "date",
            lambda x: x.dt.date.nunique()
        )
    )
    .reset_index()
)

# =========================================================
# TRANSACTION BEHAVIOR FEATURES
# =========================================================

# ---------- INFLOW ----------
inflow_behavior = (
    trx[trx["amount"] > 0]
    .groupby("client_id")
    .agg(
        avg_inflow_amount=("amount", "mean"),
        median_inflow_amount=("amount", "median"),
        max_inflow_amount=("amount", "max"),
        min_inflow_amount=("amount", "min"),
        std_inflow_amount=("amount", "std")
    )
    .reset_index()
)

# ---------- OUTFLOW ----------
outflow_behavior = (
    trx[trx["amount"] < 0]
    .groupby("client_id")
    .agg(
        avg_outflow_amount=("amount", "mean"),
        median_outflow_amount=("amount", "median"),
        max_outflow_amount=("amount", "max"),
        min_outflow_amount=("amount", "min"),
        std_outflow_amount=("amount", "std")
    )
    .reset_index()
)

# =========================================================
# MONTHLY BEHAVIOR FEATURES
# =========================================================

monthly_behavior = (
    monthly_client.groupby("client_id")
    .agg(
        avg_monthly_inflow=("monthly_inflow", "mean"),
        avg_monthly_outflow=("monthly_outflow", "mean"),

        monthly_inflow_std=("monthly_inflow", "std"),
        monthly_outflow_std=("monthly_outflow", "std"),

        max_monthly_outflow=("monthly_outflow", "max"),

        avg_monthly_transactions=(
            "monthly_transaction_count",
            "mean"
        ),

        avg_active_days_per_month=(
            "active_days",
            "mean"
        )
    )
    .reset_index()
)

# =========================================================
# ACCOUNT & TIME BEHAVIOR
# =========================================================

# ---------- ACCOUNT AGE ----------
today = trx["date"].max()

card["account_age_days"] = (
    today - card["acct_open_date"]
).dt.days

account_age = (
    card.groupby("client_id")
    .agg(
        account_age_days=("account_age_days", "mean")
    )
    .reset_index()
)

# ---------- ACTIVE MONTH ----------
active_month = (
    trx.groupby("client_id")["year_month"]
    .nunique()
    .reset_index(name="active_month_count")
)

# ---------- AVG TXN PER MONTH ----------
avg_txn_month = (
    trx.groupby("client_id")
    .size()
    .reset_index(name="total_txn")
)

avg_txn_month = avg_txn_month.merge(
    active_month,
    on="client_id",
    how="left"
)

avg_txn_month["avg_transactions_per_month"] = (
    avg_txn_month["total_txn"] /
    avg_txn_month["active_month_count"]
)

avg_txn_month = avg_txn_month[
    ["client_id", "avg_transactions_per_month"]
]

# ---------- TIME RATIOS ----------
time_features = (
    trx.groupby("client_id")
    .agg(
        night_transaction_ratio=(
            "is_night",
            "mean"
        ),

        weekend_transaction_ratio=(
            "is_weekend",
            "mean"
        )
    )
    .reset_index()
)

# =========================================================
# TREND FEATURES
# =========================================================

latest_month = monthly_client["year_month"].max()

recent_3m = monthly_client[
    monthly_client["year_month"] >= latest_month - 2
]

historical = monthly_client[
    monthly_client["year_month"] < latest_month - 2
]

recent_features = (
    recent_3m.groupby("client_id")
    .agg(
        recent_avg_outflow=("monthly_outflow", "mean"),
        recent_avg_inflow=("monthly_inflow", "mean")
    )
    .reset_index()
)

historical_features = (
    historical.groupby("client_id")
    .agg(
        historical_avg_outflow=("monthly_outflow", "mean"),
        historical_avg_inflow=("monthly_inflow", "mean")
    )
    .reset_index()
)

trend_features = recent_features.merge(
    historical_features,
    on="client_id",
    how="left"
)

trend_features["spending_growth_rate"] = (
    trend_features["recent_avg_outflow"] /
    (trend_features["historical_avg_outflow"] + 1)
)

trend_features["income_growth_rate"] = (
    trend_features["recent_avg_inflow"] /
    (trend_features["historical_avg_inflow"] + 1)
)

# =========================================================
# CASHFLOW HEALTH
# =========================================================

cashflow_health = monthly_behavior[
    [
        "client_id",
        "avg_monthly_inflow",
        "avg_monthly_outflow"
    ]
].copy()

cashflow_health["avg_monthly_net_cashflow"] = (
    cashflow_health["avg_monthly_inflow"] -
    cashflow_health["avg_monthly_outflow"]
)

cashflow_health["spend_income_ratio"] = (
    cashflow_health["avg_monthly_outflow"] /
    (cashflow_health["avg_monthly_inflow"] + 1)
)

cashflow_health["savings_ratio"] = (
    cashflow_health["avg_monthly_net_cashflow"] /
    (cashflow_health["avg_monthly_inflow"] + 1)
)

# =========================================================
# MERCHANT & MCC FEATURES
# =========================================================

# ---------- MERCHANT ----------
merchant_features = (
    trx.groupby("client_id")
    .agg(
        unique_merchant_count=(
            "merchant_id",
            "nunique"
        )
    )
    .reset_index()
)

# Favorite merchant
fav_merchant = (
    trx.groupby(
        ["client_id", "merchant_id"]
    )
    .size()
    .reset_index(name="trx_count")
)

fav_merchant = (
    fav_merchant.sort_values(
        ["client_id", "trx_count"],
        ascending=[True, False]
    )
    .drop_duplicates("client_id")
    [["client_id", "merchant_id"]]
    .rename(
        columns={
            "merchant_id": "favorite_merchant"
        }
    )
)

# Merchant diversity
merchant_diversity = (
    trx.groupby("client_id")
    .agg(
        total_transaction_count=("id", "count"),
        unique_merchant_count=("merchant_id", "nunique")
    )
    .reset_index()
)

merchant_diversity["merchant_diversity_score"] = (
    merchant_diversity["unique_merchant_count"] /
    merchant_diversity["total_transaction_count"]
)

merchant_diversity = merchant_diversity[
    ["client_id", "merchant_diversity_score","total_transaction_count"]
]

# ---------- MCC ----------
mcc_spend = (
    trx[trx["cashflow_type"] == "outflow"]
    .groupby(["client_id", "mcc_description"])
    .agg(
        total_spend=("abs_amount", "sum")
    )
    .reset_index()
)

fav_mcc = (
    mcc_spend.sort_values(
        ["client_id", "total_spend"],
        ascending=[True, False]
    )
    .drop_duplicates("client_id")
    [["client_id", "mcc_description"]]
    .rename(
        columns={
            "mcc_description": "favorite_mcc"
        }
    )
)

mcc_diversity = (
    trx.groupby("client_id")
    .agg(
        unique_mcc_count=("mcc", "nunique")
    )
    .reset_index()
)

# =========================================================
# RFM FEATURES
# =========================================================

snapshot_date = trx["date"].max()

rfm = (
    trx.groupby("client_id")
    .agg(
        recency_days=(
            "date",
            lambda x: (
                snapshot_date - x.max()
            ).days
        ),

        frequency=("id", "count"),

        monetary=("abs_amount", "sum")
    )
    .reset_index()
)

# =========================================================
# FRAUD FEATURES
# =========================================================

fraud_features = (
    trx.groupby("client_id")
    .agg(

        # =====================================
        # BASIC FRAUD
        # =====================================

        fraud_transaction_count=(
            "fraud_label",
            "sum"
        ),

        fraud_ratio=(
            "fraud_label",
            "mean"
        ),

        avg_fraud_amount=(
            "abs_amount",
            lambda x: x[
                trx.loc[x.index, "fraud_label"] == 1
            ].mean()
        ),

        # =====================================
        # ERROR COUNTS
        # =====================================

        bad_cvv_count=(
            "bad_cvv",
            "sum"
        ),

        bad_card_number_count=(
            "bad_card_number",
            "sum"
        ),

        bad_expiration_count=(
            "bad_expiration",
            "sum"
        ),

        bad_pin_count=(
            "bad_pin",
            "sum"
        ),

        bad_zipcode_count=(
            "bad_zipcode",
            "sum"
        ),

        insufficient_balance_count=(
            "insufficient_balance",
            "sum"
        ),

        technical_glitch_count=(
            "technical_glitch",
            "sum"
        )
    )
    .reset_index()
)

# =========================================
# TOTAL ERROR COUNT
# =========================================

error_cols = [
    "bad_cvv_count",
    "bad_card_number_count",
    "bad_expiration_count",
    "bad_pin_count",
    "bad_zipcode_count",
    "insufficient_balance_count",
    "technical_glitch_count"
]

fraud_features["total_error_count"] = (
    fraud_features[error_cols]
    .sum(axis=1)
)

# =========================================
# ERROR RATIOS
# =========================================

fraud_features = fraud_features.merge(

    trx.groupby("client_id")
    .agg(
        total_transaction_count=("id", "count")
    )
    .reset_index(),

    on="client_id",
    how="left"
)

# Generic error ratio
fraud_features["error_ratio"] = (
    fraud_features["total_error_count"] /
    (fraud_features["total_transaction_count"] + 1)
)

# Authentication-related errors
fraud_features["auth_error_count"] = (
    fraud_features["bad_cvv_count"] +
    fraud_features["bad_pin_count"] +
    fraud_features["bad_card_number_count"]
)

fraud_features["auth_error_ratio"] = (
    fraud_features["auth_error_count"] /
    (fraud_features["total_transaction_count"] + 1)
)

# Financial stress indicator
fraud_features["insufficient_balance_ratio"] = (
    fraud_features["insufficient_balance_count"] /
    (fraud_features["total_transaction_count"] + 1)
)

# Technical issue ratio
fraud_features["technical_glitch_ratio"] = (
    fraud_features["technical_glitch_count"] /
    (fraud_features["total_transaction_count"] + 1)
)

# =========================================
# OPTIONAL CLEANING
# =========================================

numeric_cols = fraud_features.select_dtypes(
    include=np.number
).columns

fraud_features[numeric_cols] = (
    fraud_features[numeric_cols]
    .fillna(0)
)


avg_fraud_amount = (
    trx[trx["fraud_label"] == 1]
    .groupby("client_id")
    .agg(
        avg_fraud_amount=("amount", "mean")
    )
    .reset_index()
)



In [30]:
df_user.columns

Index(['id', 'current_age', 'retirement_age', 'birth_year', 'birth_month',
       'gender', 'address', 'latitude', 'longitude', 'per_capita_income',
       'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards'],
      dtype='object')

In [31]:
# =========================================================
# PREPARE BASELINE (df_user)
# =========================================================
# 1. Rename 'id' menjadi 'client_id' agar konsisten dengan tabel fitur lainnya
# 2. Kita ambil semua kolom dari df_user tanpa menghapus apapun
df_user_prepared = df_user.rename(columns={'id': 'client_id'})

# =========================================================
# FINAL CLIENT FEATURE STORE (BASELINE: df_user)
# =========================================================

# Kita mulai dari df_user agar semua user terakomodasi
client_feature_store = (
    df_user_prepared.copy()
    
    .merge(
        monthly_behavior,
        on="client_id",
        how="left"
    )

    .merge(
        inflow_behavior,
        on="client_id",
        how="left"
    )

    .merge(
        outflow_behavior,
        on="client_id",
        how="left"
    )

    .merge(
        account_age,
        on="client_id",
        how="left"
    )

    .merge(
        active_month,
        on="client_id",
        how="left"
    )

    .merge(
        avg_txn_month,
        on="client_id",
        how="left"
    )

    .merge(
        time_features,
        on="client_id",
        how="left"
    )

    .merge(
        trend_features[
            [
                "client_id", 
                "spending_growth_rate", 
                "income_growth_rate"
            ]
        ],
        on="client_id",
        how="left"
    )

    .merge(
        cashflow_health[
            [
                "client_id", 
                "avg_monthly_net_cashflow", 
                "spend_income_ratio", 
                "savings_ratio"
            ]
        ],
        on="client_id",
        how="left"
    )

    .merge(
        fav_mcc,
        on="client_id",
        how="left"
    )

    .merge(
        mcc_diversity,
        on="client_id",
        how="left"
    )

    .merge(
        merchant_features,
        on="client_id",
        how="left"
    )

    .merge(
        fav_merchant,
        on="client_id",
        how="left"
    )

    .merge(
        merchant_diversity,
        on="client_id",
        how="left"
    )

    .merge(
        rfm,
        on="client_id",
        how="left"
    )

    .merge(
        fraud_features,
        on="client_id",
        how="left"
    )

    .merge(
        avg_fraud_amount,
        on="client_id",
        how="left"
    )
)

# =========================================================
# HANDLE NULL VALUES
# =========================================================

# 1. Untuk kolom numerik, isi dengan 0
numeric_cols = client_feature_store.select_dtypes(
    include=np.number
).columns

client_feature_store[numeric_cols] = (
    client_feature_store[numeric_cols]
    .fillna(0)
)

# 2. Untuk kolom kategorikal (seperti favorite_mcc/merchant), isi dengan 'unknown'
categorical_cols = client_feature_store.select_dtypes(
    include=['object', 'category']
).columns

client_feature_store[categorical_cols] = (
    client_feature_store[categorical_cols]
    .fillna('unknown')
)

# =========================================================
# RESULT
# =========================================================

print(client_feature_store.shape)

client_feature_store.head()

print(client_feature_store.columns.tolist())

(2000, 68)
['client_id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'avg_monthly_inflow', 'avg_monthly_outflow', 'monthly_inflow_std', 'monthly_outflow_std', 'max_monthly_outflow', 'avg_monthly_transactions', 'avg_active_days_per_month', 'avg_inflow_amount', 'median_inflow_amount', 'max_inflow_amount', 'min_inflow_amount', 'std_inflow_amount', 'avg_outflow_amount', 'median_outflow_amount', 'max_outflow_amount', 'min_outflow_amount', 'std_outflow_amount', 'account_age_days', 'active_month_count', 'avg_transactions_per_month', 'night_transaction_ratio', 'weekend_transaction_ratio', 'spending_growth_rate', 'income_growth_rate', 'avg_monthly_net_cashflow', 'spend_income_ratio', 'savings_ratio', 'favorite_mcc', 'unique_mcc_count', 'unique_merchant_count', 'favorite_merchant', 'merchant_diversity_score', 'total_transaction_count_x', 'recency

In [32]:
# =========================================================
# ADVANCED CLIENT FEATURES
# =========================================================

# =========================================
# PREP
# =========================================

trx["transaction_day"] = trx["date"].dt.date
trx["transaction_hour"] = trx["date"].dt.hour

# =========================================================
# 1. VELOCITY FEATURES
# =========================================================

# ---------- DAILY TRANSACTION VELOCITY ----------
daily_trx = (
    trx.groupby(["client_id", "transaction_day"])
    .size()
    .reset_index(name="daily_transaction_count")
)

velocity_daily = (
    daily_trx.groupby("client_id")
    .agg(
        avg_transactions_per_day=(
            "daily_transaction_count",
            "mean"
        ),

        max_transactions_per_day=(
            "daily_transaction_count",
            "max"
        )
    )
    .reset_index()
)

# ---------- HOURLY TRANSACTION VELOCITY ----------
hourly_trx = (
    trx.groupby(
        [
            "client_id",
            "transaction_day",
            "transaction_hour"
        ]
    )
    .size()
    .reset_index(name="hourly_transaction_count")
)

velocity_hourly = (
    hourly_trx.groupby("client_id")
    .agg(
        max_transactions_per_hour=(
            "hourly_transaction_count",
            "max"
        )
    )
    .reset_index()
)

velocity_features = (
    velocity_daily.merge(
        velocity_hourly,
        on="client_id",
        how="left"
    )
)

# =========================================================
# 2. LIFESTYLE SCORES
# =========================================================

# Use MCC DESCRIPTION for lifestyle tagging
trx["mcc_description"] = (
    trx["mcc_description"]
    .astype(str)
    .str.lower()
)

# ---------- MCC KEYWORDS ----------
travel_mcc = [
    "airlines",
    "hotel",
    "travel"
]

dining_mcc = [
    "restaurant",
    "fast food",
    "cafe"
]

digital_mcc = [
    "electronics",
    "online",
    "digital"
]

luxury_mcc = [
    "jewelry",
    "luxury"
]

# ---------- SPENDING ONLY ----------
spend_trx = trx[
    trx["cashflow_type"] == "outflow"
].copy()

# ---------- TOTAL SPENDING ----------
total_spending = (
    spend_trx.groupby("client_id")
    .agg(
        total_spend=("abs_amount", "sum")
    )
    .reset_index()
)

# ---------- HELPER FUNCTION ----------
def calculate_lifestyle_score(
    df,
    mcc_keywords,
    score_name
):

    filtered = df[
        df["mcc_description"].str.contains(
            "|".join(mcc_keywords),
            case=False,
            na=False
        )
    ]

    score = (
        filtered.groupby("client_id")
        .agg(
            score_amount=("abs_amount", "sum")
        )
        .reset_index()
    )

    score = score.merge(
        total_spending,
        on="client_id",
        how="left"
    )

    score[score_name] = (
        score["score_amount"] /
        (score["total_spend"] + 1)
    )

    return score[
        ["client_id", score_name]
    ]

# ---------- SCORES ----------
travel_score = calculate_lifestyle_score(
    spend_trx,
    travel_mcc,
    "travel_score"
)

dining_score = calculate_lifestyle_score(
    spend_trx,
    dining_mcc,
    "dining_score"
)

digital_score = calculate_lifestyle_score(
    spend_trx,
    digital_mcc,
    "digital_score"
)

luxury_score = calculate_lifestyle_score(
    spend_trx,
    luxury_mcc,
    "luxury_score"
)

# ---------- FINAL LIFESTYLE TABLE ----------
lifestyle_scores = (
    total_spending[["client_id"]]

    .merge(
        travel_score,
        on="client_id",
        how="left"
    )

    .merge(
        dining_score,
        on="client_id",
        how="left"
    )

    .merge(
        digital_score,
        on="client_id",
        how="left"
    )

    .merge(
        luxury_score,
        on="client_id",
        how="left"
    )
)

# =========================================================
# 3. DORMANCY FEATURES
# =========================================================

snapshot_date = trx["date"].max()

dormancy_features = (
    trx.groupby("client_id")
    .agg(
        days_since_last_transaction=(
            "date",
            lambda x: (
                snapshot_date - x.max()
            ).days
        )
    )
    .reset_index()
)

# =========================================================
# 4. ESSENTIAL SPENDING FEATURES
# =========================================================

essential_mcc = [
    "grocery",
    "utilities",
    "gas",
    "pharmacy",
    "insurance"
]

essential_spending = spend_trx[
    spend_trx["mcc_description"].str.contains(
        "|".join(essential_mcc),
        case=False,
        na=False
    )
]

essential_features = (
    essential_spending.groupby("client_id")
    .agg(
        essential_spend=("abs_amount", "sum")
    )
    .reset_index()
)

essential_features = essential_features.merge(
    total_spending,
    on="client_id",
    how="left"
)

essential_features["essential_spend_ratio"] = (
    essential_features["essential_spend"] /
    (essential_features["total_spend"] + 1)
)

essential_features = essential_features[
    ["client_id", "essential_spend_ratio"]
]

# =========================================================
# 5. CUSTOMER VALUE FEATURES
# =========================================================

customer_value = (
    trx.groupby("client_id")
    .agg(
        customer_total_monetary=("abs_amount", "sum"),
        customer_total_frequency=("id", "count")
    )
    .reset_index()
)

customer_value["customer_value_score"] = (
    customer_value["customer_total_monetary"] *
    customer_value["customer_total_frequency"]
)

customer_value = customer_value[
    [
        "client_id",
        "customer_value_score"
    ]
]

# =========================================================
# FINAL MERGE TO CLIENT FEATURE STORE
# =========================================================

client_feature_store = (

    client_feature_store

    .merge(
        velocity_features,
        on="client_id",
        how="left"
    )

    .merge(
        lifestyle_scores,
        on="client_id",
        how="left"
    )

    .merge(
        dormancy_features,
        on="client_id",
        how="left"
    )

    .merge(
        essential_features,
        on="client_id",
        how="left"
    )

    .merge(
        customer_value,
        on="client_id",
        how="left"
    )
)

# =========================================================
# FINAL CLEANING
# =========================================================

numeric_cols = client_feature_store.select_dtypes(
    include=np.number
).columns

client_feature_store[numeric_cols] = (
    client_feature_store[numeric_cols]
    .fillna(0)
)

# Optional categorical fill
categorical_cols = client_feature_store.select_dtypes(
    exclude=np.number
).columns

client_feature_store[categorical_cols] = (
    client_feature_store[categorical_cols]
    .fillna("Unknown")
)

# Remove duplicates if any
client_feature_store = (
    client_feature_store
    .drop_duplicates(subset=["client_id"])
)

# =========================================================
# RESULT
# =========================================================

print("=" * 60)
print("CLIENT FEATURE STORE READY")
print("=" * 60)

print("Shape :", client_feature_store.shape)

print("\nSample:")
display(client_feature_store.head())

print("\nColumns:")
print(client_feature_store.columns.tolist())

CLIENT FEATURE STORE READY
Shape : (2000, 78)

Sample:


,client_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,...,avg_transactions_per_day,max_transactions_per_day,max_transactions_per_hour,travel_score,dining_score,digital_score,luxury_score,days_since_last_transaction,essential_spend_ratio,customer_value_score
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,...,3.250286,11.0,7.0,0.145757,0.0,0.0,0.0,1.0,0.0,1.158557e+10
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,...,2.035298,9.0,6.0,0.139811,0.0,0.0,0.0,0.0,0.0,3.021329e+09
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,...,7.217731,19.0,7.0,0.087439,0.0,0.0,0.0,0.0,0.0,2.469833e+10
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,...,2.648261,15.0,7.0,0.219591,0.0,0.0,0.0,0.0,0.0,1.101395e+10
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,...,2.789299,10.0,5.0,0.250739,0.0,0.0,0.0,0.0,0.0,8.847454e+09



Columns:
['client_id', 'current_age', 'retirement_age', 'birth_year', 'birth_month', 'gender', 'address', 'latitude', 'longitude', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards', 'avg_monthly_inflow', 'avg_monthly_outflow', 'monthly_inflow_std', 'monthly_outflow_std', 'max_monthly_outflow', 'avg_monthly_transactions', 'avg_active_days_per_month', 'avg_inflow_amount', 'median_inflow_amount', 'max_inflow_amount', 'min_inflow_amount', 'std_inflow_amount', 'avg_outflow_amount', 'median_outflow_amount', 'max_outflow_amount', 'min_outflow_amount', 'std_outflow_amount', 'account_age_days', 'active_month_count', 'avg_transactions_per_month', 'night_transaction_ratio', 'weekend_transaction_ratio', 'spending_growth_rate', 'income_growth_rate', 'avg_monthly_net_cashflow', 'spend_income_ratio', 'savings_ratio', 'favorite_mcc', 'unique_mcc_count', 'unique_merchant_count', 'favorite_merchant', 'merchant_diversity_score', 'total_transaction_count_x', 'recency_

In [33]:
client_feature_store.to_csv("data\client_feature_store-v2.csv", index_label= False)

In [34]:
client_feature_store.to_csv(
    "feature_store/client_feature_store-v2.csv",
    index=False
)

In [3]:
client_feature_store = pd.read_csv(
    "../feature_store/client_feature_store-v2.csv"
)



In [6]:
client_feature_store['yearly_income'] = client_feature_store['yearly_income'].str.replace("$","")

In [7]:
client_feature_store_filtered = client_feature_store[(client_feature_store['yearly_income'] != 0 ) & (client_feature_store['savings_ratio'] != 0 ) ]
client_feature_store_filtered.head()

,client_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,...,avg_transactions_per_day,max_transactions_per_day,max_transactions_per_hour,travel_score,dining_score,digital_score,luxury_score,days_since_last_transaction,essential_spend_ratio,customer_value_score
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,...,3.250286,11.0,7.0,0.145757,0.0,0.0,0.0,1.0,0.0,1.158557e+10
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,...,2.035298,9.0,6.0,0.139811,0.0,0.0,0.0,0.0,0.0,3.021329e+09
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,...,7.217731,19.0,7.0,0.087439,0.0,0.0,0.0,0.0,0.0,2.469833e+10
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,...,2.648261,15.0,7.0,0.219591,0.0,0.0,0.0,0.0,0.0,1.101395e+10
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,...,2.789299,10.0,5.0,0.250739,0.0,0.0,0.0,0.0,0.0,8.847454e+09


In [8]:
client_feature_store_filtered.shape

(1219, 78)

In [9]:
sample_df = client_feature_store_filtered.sample(500)



In [10]:
sample_df.to_csv(
    "../feature_store/sample_feature_store.csv",
    index=False
)